# Blog Search Agent — Mode Testing

Interactive notebook to test both modes of the blog search agent:

1. **Chat mode** — conversational, free-form natural language, multi-turn capable
2. **Workflow mode** — deterministic pipeline, structured JSON output with Pydantic validation

### Prerequisites
- AWS credentials configured (`AWS_PROFILE` or env vars)
- `.env` file with `GATEWAY_URL`, `COGNITO_*` vars (or `GATEWAY_TOKEN`)
- `uv sync` to install dependencies

In [1]:
import sys
import os
import json
import asyncio
import logging
from pathlib import Path

# Ensure blog_search is importable
sys.path.insert(0, str(Path.cwd()))

# Force suppress all INFO/DEBUG logs — only show ERROR
logging.basicConfig(level=logging.ERROR, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s", force=True)
logging.getLogger().setLevel(logging.ERROR)

In [2]:
# # One-time setup: Create a memory resource (run once, then add MEMORY_ID to .env)
# from bedrock_agentcore.memory.controlplane import MemoryControlPlaneClient

# memory_client = MemoryControlPlaneClient(region_name="us-east-1")
# memory = memory_client.create_memory(
#     name="blogSearchSessionMemory",
#     description="Short-term session memory for blog search agent",
#     event_expiry_days=30,
#     wait_for_active=True,
# )
# print(f"Memory created! Add this to your .env file:")
# print(f"MEMORY_ID={memory['id']}")

## Configuration

- **Chat mode**: `CHAT_LOOKBACK_DAYS` — how many days back to search for events. The agent then searches for events in a time-window that spans from (today-chat_lookback_days) to current_date.
- **Workflow mode**: `WORKFLOW_LOOKBACK_HOURS` — how many hours back from `WORKFLOW_REFERENCE_DATE` to search
- **Reference date**: Set to a past date (e.g. `"2026-07-15"`) for testing, or `""` for real current date

In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CONFIGURATION — Change these values to control test behavior
# ═══════════════════════════════════════════════════════════════════════

# Chat mode: how many days back to search
CHAT_LOOKBACK_DAYS = 14

# Workflow mode: how many hours back from reference_date to search
WORKFLOW_LOOKBACK_HOURS = 24  # 0 = reference_date only, 24 = past day, 168 = past week

# Debug mode — prints full tool traces and system prompts
DEBUG = False

print(f"Chat lookback: {CHAT_LOOKBACK_DAYS} days")
print(f"Workflow lookback: {WORKFLOW_LOOKBACK_HOURS}")
print(f"Debug: {DEBUG}")

Chat lookback: 14 days
Workflow lookback: 24
Debug: False


In [4]:
from blog_search_agent import (
    create_chat_agent,
    run_blog_search_workflow,
    extract_json_from_response,
)
from models import EVENT_TYPES, validate_response

print(f"Event types: {EVENT_TYPES}")

Event types: ['INJURY', 'CANCELLATION', 'VENUE_CHANGE', 'SCHEDULE_CHANGE', 'ROSTER']


In [5]:
async def stream_response(agent, input_query):
    async for event in agent.stream_async(input_query):
        if "data" in event:
            print(event["data"], end="", flush=True)

---
## 1. Chat Mode

Chat mode is conversational — it accepts free-form questions and responds with cited, grounded answers.
The agent has access to: `registry_lookup`, `rss_fetch`, `web_fetch`, `gateway__WebSearch`.

## Create a chat agent with streaming output
chat_agent = create_chat_agent(lookback_days=CHAT_LOOKBACK_DAYS, debug=DEBUG)

## Stream the response — tokens appear as they're generated
print("Streaming response:\n")
async for event in chat_agent.stream_async("Find any recent injury news for Colgate Raiders basketball"):
    if "data" in event:
        print(event["data"], end="", flush=True)
print()

In [6]:
# Create a chat agent (no memory — local testing without AgentCore session)
chat_agent = create_chat_agent(lookback_days=CHAT_LOOKBACK_DAYS, debug=False)
input_query = "Find any recent injury news for Colgate Raiders basketball"
# Ask about recent events for a team
print("Streaming response:\n")
await stream_response(chat_agent, input_query)


Streaming response:

Looking up blog sources for Colgate Raiders...Found 4 sources. Checking RSS feeds for recent posts...No RSS results found. Searching the web for recent injury news...Searching the web for Colgate Raiders men's basketball injury updates July 2026...---

## 🏀 Colgate Raiders Men's Basketball — Injury Report (July 3–17, 2026)

**No injury events were found** for Colgate Raiders men's basketball within the current search window (July 3–17, 2026).

Here's a summary of what was checked:
| Source | Result |
|---|---|
| Colgate Athletics Official Site (RSS) | No injuries found |
| The Colgate Maroon News (RSS) | No injuries found |
| RaidersTalk Fan Blog | No injuries found |
| Colgate Athletics Men's Basketball Page | No injuries found |
| Yahoo Sports NCAAB Injuries | No injuries found |

### 📌 Notable Context (Outside Search Window)
While no injuries were found in the current window, the web search did surface some relevant **non-injury** news:
- **Schedule Note**: Colg

### 1.2 Follow-up question (same agent instance = same conversation context)

In [7]:
# Follow-up on the same agent — tests multi-turn within a session

await stream_response(chat_agent, "okay, then give the exact snippets and sources for roster change?")

Looking up roster news for Colgate Raiders — fetching the incoming class article directly...Here are the exact extracted roster events and their source snippets:

---

## 🏀 Colgate Raiders — Roster Events

> ⚠️ **Note:** These events were published on **June 16, 2026**, which is just outside the July 3–17 search window. They are included here as they were surfaced during the search and represent the most recent roster news available.

**Source:** [Colgate Athletics — Men's Basketball Announces Incoming Class](https://colgateathletics.com/news/2026/6/15/mens-basketball-mens-basketball-announces-incoming-class.aspx)
**Published:** June 16, 2026

---

### 1. Jayden Harrington — ROSTER ADD
> *"Colgate University Head Men's Basketball Coach Matt Langel has announced three additions to the Raiders' roster for the upcoming 2026-27 season. The incoming class – comprising two first-years and a sophomore transfer – adds versatile guard depth from three different states."*

- **Position:** G | **

### 1.3 Out-of-scope request

In [8]:
# Out-of-scope: should respond politely and explain what it can help with
await stream_response(chat_agent, "What's the weather like in New York today?")


I specialize in **NCAA Men's Basketball event extraction** and am not able to help with weather queries. 🏀

I can help you find the following for any NCAA Men's Basketball team:
- 🤕 **Injury news**
- 📅 **Schedule changes**
- 🏟️ **Venue updates**
- 📋 **Roster moves**
- ❌ **Game cancellations**

Would you like me to search for any of these for a specific team?

### 1.4 Greeting (conversational handling)

In [9]:
# Greeting: should respond conversationally, not attempt tool calls
await stream_response(chat_agent, "Hi! How are you doing today?")

I'm doing great, thanks for asking! 😊 However, I'm a specialized **NCAA Men's Basketball event extraction agent**, so I'm best put to work finding sports intelligence for traders!

Here's what I can help you with:
- 🤕 **Injury news** — player injuries, recovery timelines
- 📅 **Schedule changes** — postponements, rescheduled games
- 🏟️ **Venue updates** — arena changes
- 📋 **Roster moves** — transfers, additions, departures
- ❌ **Cancellations** — game cancellations

Just give me a **team name** and I'll get to work! Which NCAA Men's Basketball team would you like me to look into?

### 1.5 Specific event type request

In [10]:
# Ask for a specific event type in user-provided URLs
specific_agent = create_chat_agent(lookback_days=CHAT_LOOKBACK_DAYS, debug=DEBUG)
query="""Are there any roaster news or injury news for Duke Blue Devils 
in https://balldurham.com/ or 
https://forums.dukebasketballreport.com/index.php?forums/elizabeth-king-forum.2/ in the past 2 months?"""
await stream_response(chat_agent, query)


The user has provided specific URLs, so I'll fetch those directly. Also, the user asked for the **past 2 months** — I'll use **2026-05-17 to 2026-07-17** as the date range.

Reading article at https://balldurham.com/...
Reading article at https://forums.dukebasketballreport.com/index.php?forums/elizabeth-king-forum.2/...Here are the results from both sources for **Duke Blue Devils** (May 17 – July 17, 2026):

---

## 🏀 Duke Blue Devils — Roster & Injury News (Past 2 Months)

### Source 1: [BallDurham.com](https://balldurham.com/)
**No roster or injury events found** within the date range.

---

### Source 2: [Duke Basketball Report — Elizabeth King Forum](https://forums.dukebasketballreport.com/index.php?forums/elizabeth-king-forum.2/)

#### 1. 📋 ROSTER — Kager Knueppel
> *"Welcome to Duke, Kager Knueppel!"*

- **Date:** June 29, 2026
- **Summary:** Kager Knueppel joined the Duke Blue Devils roster.

---

#### 2. 📋 ROSTER — 2026-27 Roster & Jerseys
> *"MBB: 2026-27 Roster and Jerseys"*

# AgentCore Memory Test

In [17]:
# Session-scoped memory test — requires MEMORY_ID in .env
# Uses a fixed session_id so you can re-run and see history restored

SESSION_ID = "test-session-004"
USER_ID = "test-user-local"

# Turn 1: Ask a question
memory_agent = create_chat_agent(
    user_id=USER_ID,
    session_id=SESSION_ID,
    lookback_days=CHAT_LOOKBACK_DAYS,
    debug=DEBUG,
)
question = "Find any injury news for Duke Blue Devils"
print(f"turn 1 question: {question}\nResponse: ")
await stream_response(memory_agent, question)


# Turn 2: Follow-up (same session — history should be available)
followup_question = "what about Vermont Catamounts?"
print(f"\nTURN 2 (follow-up): {followup_question}\nResponse: ")
await stream_response(memory_agent, followup_question)

turn 1 question: Find any injury news for Duke Blue Devils
Response: 
Looking up blog sources for Duke Blue Devils...Found 2 RSS feeds and several blog sources. Checking RSS feeds and searching the web simultaneously...I see some potentially relevant results. Let me check the most promising articles more closely — particularly the July 2026 pieces.No injury events were extracted from those articles. Let me also check the remaining active blog sources directly.---

## 🏀 Duke Blue Devils — Injury News (July 3–17, 2026)

After a thorough search across all available sources — including:
- **Duke Basketball Report** RSS feed
- **Pine State Playbook** (Substack) RSS feed
- **Ball Durham** (SB Nation)
- **GoDuke.com** (Official site)
- **Web search** across SI.com, Sporting News, and other outlets

**No injury events for Duke Blue Devils men's basketball were found within the July 3–17, 2026 date window.**

### 📝 Notes:
- The most recent injury news found in search results dates back to **Mar

In [18]:
# Cross-instance test: NEW agent instance, SAME session_id
# If memory is working, this agent should "remember" the previous conversation
memory_agent_2 = create_chat_agent(
    user_id=USER_ID,
    session_id=SESSION_ID,  # same session — should restore history
    lookback_days=CHAT_LOOKBACK_DAYS,
    debug=DEBUG,
)
followup_question_2 = "What did we discuss earlier about Duke?"
print(f"\nTURN 3 (follow-up with new agent instance, same session_id): {followup_question}\nResponse: ")
await stream_response(memory_agent_2, followup_question_2)


TURN 3 (follow-up with new agent instance, same session_id): what about Vermont Catamounts?
Response: 
Based on our earlier conversation, here's what was found regarding the **Duke Blue Devils**:

## 🏀 Duke Blue Devils — Injury Search Summary

**Date Range Searched:** July 3–17, 2026

**Sources Checked:**
- Duke Basketball Report RSS feed
- Pine State Playbook (Substack) RSS feed
- Ball Durham (SB Nation)
- GoDuke.com (Official site)
- Web search across SI.com, Sporting News, and other outlets

### Result: No Injuries Found in the Date Window
No injury events for Duke Blue Devils men's basketball were found within the **July 3–17, 2026** date range.

### 📝 Notable Context (Outside Date Range)
Two injuries were surfaced from **March 2026** (NCAA Tournament period) but fell outside the requested window:
1. **Patrick Ngongba II** — Foot soreness; was being held out cautiously ahead of the NCAA Tournament Round of 64 vs. Siena.
2. **Caleb Foster** — Underwent surgery; timeline made a retu

---
## 2. Workflow Mode

Workflow mode is a deterministic pipeline for batch/scheduled runs:

 Step 1: Registry lookup (plain Python, no LLM)
    Fetches team's blog URLs from DynamoDB

    Only returns crawlable + non-outdated URLs

Step 2: RSS feeds — parallel, no agent loop

    All RSS URLs are called concurrently via asyncio.gather

    Each rss_fetch call: fetches the feed → Nova 2 Lite extracts events → returns structured events

    Events are validated against EventResult Pydantic schema; malformed ones are dropped

Step 3: Non-RSS URLs — agent loop (sequential reasoning, parallel tool execution)

    A Sonnet agent is created with gateway__WebSearch + web_fetch tools
    
    The agent decides search queries and which URLs to fetch (requires LLM reasoning)

    When the agent requests multiple web_fetch calls in one turn, Strands executes them concurrently (via ConcurrentToolExecutor)

    But each agent "turn" (think → call tools → think) is sequential

Step 4: Aggregate all events and return

So: RSS is fully parallel (all feeds at once). Non-RSS is agent-driven — parallel within a single turn, but sequential across turns.

### 2.1 Standard workflow run (today only — default)

In [13]:
result = await run_blog_search_workflow(
    team="Duke Blue Devils",
    lookback_hours=WORKFLOW_LOOKBACK_HOURS,
    reference_date="2026-03-10",
    debug=False,
)

print("\n" + "="*70)
print("WORKFLOW RESULT:")
print("="*70)
print(json.dumps(result, indent=2, default=str))


WORKFLOW RESULT:
{
  "status": "success",
  "mode": "workflow",
  "team": "Duke Blue Devils",
  "sport": "CBB",
  "lookback_hours": 24,
  "date_range": {
    "start": "2026-03-09",
    "end": "2026-03-10"
  },
  "results": [
    {
      "sport": "CBB",
      "team": "Duke Blue Devils",
      "event_type": "INJURY",
      "player_name": "Caleb Foster",
      "excerpt": "Duke starting point guard Caleb Foster injured his right foot in the first half of the Blue Devils' 76-61 win over North Carolina to close out the regular season on Saturday. At his Tuesday morning media availability, Scheyer clarified that Foster suffered a fracture in his right foot and is out for a \"foreseeable\" time after receiving surgery.",
      "summary": "Caleb Foster suffered a right foot fracture and underwent surgery, ruling him out for the foreseeable future with a possible Final Four return.",
      "source_url": "https://balldurham.com/jon-scheyer-gives-new-caleb-foster-update-that-changes-everything-fo

### 2.2 Validate the output schema

In [14]:
if result.get("status") == "success":
    # Re-validate the returned data against Pydantic models
    validation = validate_response({
        "results": result["results"],
        "retrieval_diagnostics": result["retrieval_diagnostics"],
    })
    if isinstance(validation, list):
        print("VALIDATION ERRORS:")
        for err in validation:
            print(f"  - {err}")
    else:
        print(f"Validation PASSED: {len(validation.results)} events")
        for ev in validation.results:
            print(f"  [{ev.event_type}] {ev.summary[:80]}")
else:
    print(f"Workflow returned error: {result.get('error')}")

Validation PASSED: 9 events
  [INJURY] Caleb Foster suffered a right foot fracture and underwent surgery, ruling him ou
  [INJURY] Duke point guard Caleb Foster suffered a right foot fracture requiring surgery a
  [INJURY] Duke's Pat Ngongba II is out for the ACC Tournament due to foot soreness with ho
  [INJURY] Caleb Foster suffered a right foot fracture and will miss at least several weeks
  [INJURY] Patrick Ngongba ruled out for the ACC Tournament with right foot soreness but ex
  [INJURY] Patrick Ngongba II will miss the ACC Tournament with foot soreness.
  [INJURY] Caleb Foster will miss indefinite time after suffering a right foot fracture req
  [INJURY] Caleb Foster suffered a fractured bone in his foot and will be out for the fores
  [INJURY] Patrick Ngongba will miss the ACC Tournament as a precautionary measure.


### 2.5 Workflow with non-existent team (error path)

In [15]:
result_err = await run_blog_search_workflow(
    team="Nonexistent University Falcons",
    sport="NCAA Men's Basketball",
    lookback_hours=WORKFLOW_LOOKBACK_HOURS,
    reference_date="",
    debug=DEBUG,
)

print(f"Status: {result_err.get('status')}")
print(f"Error: {result_err.get('error', 'N/A')}")

Status: error
Error: Team 'Nonexistent University Falcons' not found in registry


---
## 4. JSON Extraction (Unit Tests)

Test the balanced-brace JSON extraction from agent response text.

In [16]:
# JSON embedded in markdown code block
text_with_codeblock = '''Here are the results:
```json
{"results": [], "retrieval_diagnostics": {"urls_total": 0, "urls_with_rss": 0, "urls_without_rss": 0, "web_searches_performed": 0, "rss_feeds_fetched": 0, "web_fetches_performed": 0}}
```
'''
parsed = extract_json_from_response(text_with_codeblock)
assert parsed is not None
assert parsed["results"] == []
print("Code block extraction: PASSED")

# Raw JSON (no wrapping)
raw_json = '{"results": [{"sport": "Basketball"}], "retrieval_diagnostics": {"urls_total": 1, "urls_with_rss": 0, "urls_without_rss": 1, "web_searches_performed": 0, "rss_feeds_fetched": 0, "web_fetches_performed": 0}}'
parsed = extract_json_from_response(raw_json)
assert parsed is not None
assert len(parsed["results"]) == 1
print("Raw JSON extraction: PASSED")

# JSON with preamble text
text_with_preamble = 'Based on my search, here is the result: {"results": [], "retrieval_diagnostics": {"urls_total": 2, "urls_with_rss": 1, "urls_without_rss": 1, "web_searches_performed": 1, "rss_feeds_fetched": 1, "web_fetches_performed": 0}}'
parsed = extract_json_from_response(text_with_preamble)
assert parsed is not None
assert parsed["retrieval_diagnostics"]["urls_total"] == 2
print("Preamble extraction: PASSED")

# Non-JSON response (conversational)
conversational = "I couldn't find any recent injury news for that team. Would you like me to check another team?"
parsed = extract_json_from_response(conversational)
assert parsed is None
print("Non-JSON response: correctly returned None")

Code block extraction: PASSED
Raw JSON extraction: PASSED
Preamble extraction: PASSED
Non-JSON response: correctly returned None
